# Physics-constrained Deep Learning for High-dimensional Surrogate Modeling and Uncertainty Quantification without Labeled Data

**Paper:** Zhu, Y., Zabaras, N., Koutsourelakis, P.-S., Perdikaris, P. (2019). *Physics-constrained deep learning for high-dimensional surrogate modeling and uncertainty quantification without labeled data.* Journal of Computational Physics, 394, 56-81.

**Carpeta origen:** `PINNs/4. Otros/Physics-constrained-deep-learning-for-high-dimensional-s_2019_Journal-of-Comp.pdf`

## Como se usan las PINNs en este paper

El paper aborda el modelado sustituto (*surrogate*) de EDPs con **entrada de alta dimension** (un campo de permeabilidad $K(s)$ completo, no solo unos pocos parametros escalares). Su contribucion central (Seccion 3.2) es entrenar la red **sin ningun dato etiquetado de la solucion** $u$: en vez de una perdida de regresion supervisada $\|\mathbf{y}-\hat{\mathbf{y}}_\theta(\mathbf{x})\|^2$ (Eq. 9, lo que llaman *data-driven surrogate*, DDS), usan directamente la **perdida fisica** (Eq. 8):

$$L(\theta;\{\mathbf{x}^{(i)}\})=\frac{1}{N}\sum_i\Big[V\big(\hat{\mathbf{y}}_\theta(\mathbf{x}^{(i)}),\mathbf{x}^{(i)}\big)+\lambda B\big(\hat{\mathbf{y}}_\theta(\mathbf{x}^{(i)})\big)\Big]$$

donde $V$ es la **perdida de residuo de la EDP** (Eq. 10, para el problema de flujo de Darcy $-\nabla\cdot(K\nabla u)=f$ del paper):

$$V(u;K)=\int_\mathcal{S}\big(\nabla\cdot(K\nabla u)+f\big)^2\,ds$$

y $B$ es la perdida de condiciones de contorno. Esto es exactamente el mecanismo de una PINN (residuo de la EDP + condiciones de contorno, sin datos de $u$), aplicado aqui a una red que recibe como entrada el **campo $K(s)$ completo** (llaman a esto *physics-constrained surrogate*, PCS). El paper compara esta perdida fisica pura contra el enfoque supervisado clasico (DDS, que si necesita simulaciones previas costosas para generar $\{y^{(i)}\}$), mostrando que **PCS logra precision comparable sin necesitar ninguna simulacion de referencia para entrenar**.

Este cuaderno reproduce fielmente el **problema modelo del paper** (Seccion 4, flujo de Darcy 2D, Eq. 2-3): $-\nabla\cdot(K\nabla u)=0$ en $[0,1]^2$, con Dirichlet $u=1$ en $x=0$, $u=0$ en $x=1$, y Neumann de flujo nulo en $y=0,1$, entrenando una red **puramente con la perdida fisica $V(u;K)$ (Eq. 10), sin ningun dato de $u$**, y validando el resultado (solo con fines de verificacion, nunca usado en el entrenamiento) contra una solucion de referencia por diferencias finitas.

**Simplificacion declarada:** el paper usa una red **decodificadora convolucional** (Fig. 1) que recibe el campo $K$ completo como imagen y usa filtros de Sobel para aproximar derivadas espaciales en la cuadricula &mdash; una arquitectura mas eficiente para entradas de muy alta dimension. Aqui usamos, en su lugar, una **red totalmente conectada** con coordenadas $(x,y)$ como entrada (la alternativa que el propio paper discute en la Eq. 6 como el enfoque "FC-NN" mas simple), evaluando las derivadas exactas por diferenciacion automatica en vez de filtros de Sobel &mdash; el mismo principio de "perdida fisica sin datos etiquetados", con una arquitectura mas simple de reproducir.

## Repositorio publico

El paper **incluye explicitamente** su repositorio de codigo (Seccion 4): "The code and datasets for this work are available at https://github.com/cics-nd/pde-surrogate".

- **cics-nd/pde-surrogate** &mdash; https://github.com/cics-nd/pde-surrogate

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Problema modelo (Seccion 4): flujo de Darcy 2D con campo de permeabilidad log-Gaussiano

In [ ]:
def K_field(x, y):
    """Campo de log-permeabilidad sintetico (analogo a una muestra de campo aleatorio
    Gaussiano truncado por Karhunen-Loeve, GRF KLE, usado como entrada en el paper)."""
    g = 0.6 * torch.sin(2 * np.pi * x) * torch.cos(2 * np.pi * y) + \
        0.3 * torch.cos(4 * np.pi * x + 1.0) * torch.sin(2 * np.pi * y)
    return torch.exp(g)

n_plot = 80
xs = np.linspace(0, 1, n_plot)
ys = np.linspace(0, 1, n_plot)
Xg, Yg = np.meshgrid(xs, ys)
K_plot = K_field(torch.tensor(Xg), torch.tensor(Yg)).numpy()
plt.figure(figsize=(5, 4))
plt.imshow(K_plot, origin='lower', extent=[0, 1, 0, 1], cmap='viridis')
plt.colorbar(label='K(x,y)')
plt.title('Campo de permeabilidad de entrada K(x,y)')
plt.show()

## 2. Solucion de referencia (SOLO para validacion posterior, diferencias finitas -- NUNCA usada en el entrenamiento)

In [ ]:
def solve_darcy_fd(n=61):
    xs_fd = np.linspace(0, 1, n)
    ys_fd = np.linspace(0, 1, n)
    h = xs_fd[1] - xs_fd[0]
    Xf, Yf = np.meshgrid(xs_fd, ys_fd, indexing='ij')
    K_np = K_field(torch.tensor(Xf), torch.tensor(Yf)).numpy()

    idx = lambda i, j: i * n + j
    A = np.zeros((n * n, n * n))
    b = np.zeros(n * n)

    for i in range(n):
        for j in range(n):
            k = idx(i, j)
            if i == 0:
                A[k, k] = 1.0; b[k] = 1.0             # Dirichlet u=1 en x=0
            elif i == n - 1:
                A[k, k] = 1.0; b[k] = 0.0             # Dirichlet u=0 en x=1
            elif j == 0:
                A[k, idx(i, 0)] = -1; A[k, idx(i, 1)] = 1  # Neumann flujo nulo en y=0
            elif j == n - 1:
                A[k, idx(i, n - 1)] = 1; A[k, idx(i, n - 2)] = -1  # Neumann flujo nulo en y=1
            else:
                Kp = 0.5 * (K_np[i + 1, j] + K_np[i, j]); Km = 0.5 * (K_np[i, j] + K_np[i - 1, j])
                Kt = 0.5 * (K_np[i, j + 1] + K_np[i, j]); Kb = 0.5 * (K_np[i, j] + K_np[i, j - 1])
                A[k, idx(i + 1, j)] = Kp / h**2
                A[k, idx(i - 1, j)] = Km / h**2
                A[k, idx(i, j + 1)] = Kt / h**2
                A[k, idx(i, j - 1)] = Kb / h**2
                A[k, k] = -(Kp + Km + Kt + Kb) / h**2
    u_fd = np.linalg.solve(A, b).reshape(n, n)
    return xs_fd, ys_fd, u_fd

xs_fd, ys_fd, u_ref = solve_darcy_fd()
print('Referencia FD calculada (61x61), usada solo para verificar el resultado, no para entrenar.')

## 3. Surrogate fisicamente restringido (PCS): red totalmente conectada, entrenada SIN datos de u (Eq. 8, 10)

In [ ]:
class PCS(nn.Module):
    def __init__(self, n_hidden=5, n_neurons=64):
        super().__init__()
        layers = [nn.Linear(2, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, xy):
        raw = self.net(xy)
        x = xy[:, 0:1]
        # restriccion dura de Dirichlet: u(0,y)=1, u(1,y)=0, interpolando linealmente + correccion de la red
        return (1 - x) + x * (1 - x) * raw


model = PCS().to(device)


def d_d(f, v, idx):
    g = torch.autograd.grad(f, v, grad_outputs=torch.ones_like(f),
                             create_graph=True, retain_graph=True)[0]
    return g[:, idx:idx + 1]

## 4. Perdida $V(u;K)$ (Eq. 10) + Neumann en y=0,1 -- unicamente fisica, sin etiquetas de u

In [ ]:
N_col = 3000
xy_col = torch.rand(N_col, 2, device=device).requires_grad_(True)

N_b = 300
x_b = torch.rand(N_b, 1, device=device)
y0_pts = torch.cat([x_b, torch.zeros(N_b, 1, device=device)], dim=1).requires_grad_(True)
y1_pts = torch.cat([x_b, torch.ones(N_b, 1, device=device)], dim=1).requires_grad_(True)


def compute_loss(model, lam_bc=10.0):
    xy = xy_col
    u = model(xy)
    Kxy = K_field(xy[:, 0:1], xy[:, 1:2])
    u_x = d_d(u, xy, 0)
    u_y = d_d(u, xy, 1)
    flux_x = Kxy * u_x
    flux_y = Kxy * u_y
    div_flux = d_d(flux_x, xy, 0) + d_d(flux_y, xy, 1)
    V = torch.mean(div_flux**2)  # Eq. (10), f=0 en el problema modelo del paper

    u_b0 = model(y0_pts); u_b1 = model(y1_pts)
    u_b0_y = d_d(u_b0, y0_pts, 1)
    u_b1_y = d_d(u_b1, y1_pts, 1)
    B = torch.mean(u_b0_y**2) + torch.mean(u_b1_y**2)  # Neumann: du/dy=0 en y=0,1

    return V + lam_bc * B, V.item(), B.item()

## 5. Entrenamiento

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
history = []
for epoch in range(4000):
    optimizer.zero_grad()
    loss, v_loss, b_loss = compute_loss(model)
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e} | V={v_loss:.4e} | B={b_loss:.4e}')

## 6. Validacion posterior (solo para verificar, no usada en el entrenamiento) contra la referencia FD

In [ ]:
Xf, Yf = np.meshgrid(xs_fd, ys_fd, indexing='ij')
xy_test = torch.tensor(np.stack([Xf.ravel(), Yf.ravel()], axis=1), dtype=torch.float32, device=device)
with torch.no_grad():
    u_pred = model(xy_test).cpu().numpy().reshape(Xf.shape)

err = 100 * np.linalg.norm(u_pred - u_ref) / np.linalg.norm(u_ref)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(u_ref.T, origin='lower', extent=[0, 1, 0, 1], cmap='RdBu_r')
axes[0].set_title('u(x,y) referencia (diferencias finitas)'); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(u_pred.T, origin='lower', extent=[0, 1, 0, 1], cmap='RdBu_r')
axes[1].set_title('u(x,y) surrogate (sin datos etiquetados)'); plt.colorbar(im1, ax=axes[1])
im2 = axes[2].imshow(np.abs(u_pred - u_ref).T, origin='lower', extent=[0, 1, 0, 1], cmap='inferno')
axes[2].set_title('|error|'); plt.colorbar(im2, ax=axes[2])
plt.tight_layout()
plt.show()

print(f'Error relativo L2 (surrogate sin datos etiquetados vs. referencia FD): {err:.2f}%')